# LLM-gestützte Konsistenzanalyse (DeepSeek R1 Zero)

Dieses Notebook kombiniert die in den vorherigen Schritten extrahierten Claims (`data/claims.csv`) und KPIs (`data/kpis.csv`). Anschließend wird das offene Modell **DeepSeek R1 Zero** verwendet, um mögliche Widersprüche bzw. Greenwashing-Indikatoren zu bewerten.

## 1. Setup

Das Notebook setzt auf `pandas` für die Datenverarbeitung sowie `transformers` und `torch` für das LLM. Stelle sicher, dass ausreichend GPU-Speicher zur Verfügung steht (das Modell umfasst mehrere Milliarden Parameter).

In [ ]:
import json
from pathlib import Path
from typing import Dict, List

import pandas as pd
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
import torch

DATA_DIR = Path("data")
claims_path = DATA_DIR / "claims.csv"
kpis_path = DATA_DIR / "kpis.csv"

claims_df = pd.read_csv(claims_path)
kpis_df = pd.read_csv(kpis_path)
claims_df.head(), kpis_df.head()

## 2. Kontextaufbereitung

Pro Claim werden die thematisch nächstliegenden KPIs gesucht. Die Zuordnung basiert auf dem Dokumentnamen und der Seitennähe.

In [ ]:
def gather_kpis_for_claim(claim_row, max_kpis: int = 5) -> List[Dict[str, str]]:
    doc_kpis = kpis_df[kpis_df['document_id'] == claim_row['document_id']].copy()
    if doc_kpis.empty:
        return []
    doc_kpis['page_distance'] = (doc_kpis['page'] - claim_row['page']).abs()
    doc_kpis = doc_kpis.sort_values(['page_distance', 'metric'])
    return doc_kpis.head(max_kpis)[['metric', 'value', 'context', 'page']].to_dict(orient='records')

sample_claim = claims_df.iloc[0] if not claims_df.empty else None
gather_kpis_for_claim(sample_claim) if sample_claim is not None else []

## 3. Prompt-Generierung

Claims und KPIs werden zu einem strukturierten Prompt zusammengeführt. Der Prompt fordert das Modell auf, Konsistenz, mögliche Greenwashing-Indikatoren und eine Begründung zu liefern.

In [ ]:
def build_prompt(claim: Dict[str, str], kpis: List[Dict[str, str]]) -> str:
    kpi_lines = []
    for idx, entry in enumerate(kpis, start=1):
        kpi_lines.append(f"{idx}. Metric: {entry['metric']} | Value: {entry['value']} | Page: {entry['page']} | Context: {entry['context']}")
    kpi_block = '
'.join(kpi_lines) if kpi_lines else 'Keine zugehörigen KPIs gefunden.'
    prompt = f"Du bist ein Analyst, der Nachhaltigkeitsbehauptungen mit Finanz- und Emissionskennzahlen abgleicht.

Claim (Seite {claim['page']}): {claim['sentence']}

Relevante KPIs:
{kpi_block}

Aufgabe:
1. Bewerte, ob der Claim durch die KPIs gestützt wird.
2. Weisen auf mögliche Widersprüche oder Greenwashing-Risiken hin.
3. Vergib eine Bewertung (Supported, Inconsistent, Unclear).
4. Gib eine kurze Begründung und führe genutzte KPIs an.

Antworte im JSON-Format mit den Schlüsseln `assessment`, `confidence` und `rationale`."
        return prompt

build_prompt(sample_claim, gather_kpis_for_claim(sample_claim)) if sample_claim is not None else ''

## 4. Laden des DeepSeek R1 Zero Modells

Der folgende Abschnitt lädt das offene DeepSeek R1 Zero Modell. Passe ggf. `device_map` und `torch_dtype` an die verfügbare Hardware an. Für reine CPU-Ausführung kann `device_map=None` verwendet werden (deutlich langsamer).

In [ ]:
model_name = "deepseek-ai/DeepSeek-R1-Zero"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto",
)
generation_pipeline = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    max_new_tokens=512,
    temperature=0.1,
    do_sample=False
)

## 5. Bewertung der Claims

Für Demonstrationszwecke kann die Anzahl der Claims reduziert werden (z. B. `max_claims = 10`). Die Ergebnisse werden als JSON-Liste in `data/evaluations.json` gespeichert.

In [ ]:
max_claims = min(10, len(claims_df))
evaluations = []
for _, row in claims_df.head(max_claims).iterrows():
    kpis = gather_kpis_for_claim(row)
    prompt = build_prompt(row, kpis)
    response = generation_pipeline(prompt)[0]['generated_text']
    evaluations.append({
        "document_id": row['document_id'],
        "claim_page": int(row['page']),
        "claim_text": row['sentence'],
        "kpis": kpis,
        "model_response": response
    })

evaluations[:2]

## 6. Export der Ergebnisse

Die Modellantworten werden in einer JSON-Datei abgelegt, die für qualitative Auswertungen (z. B. manuelle Validierung oder Visualisierung) genutzt werden kann.

In [ ]:
evaluations_path = DATA_DIR / "evaluations.json"
with open(evaluations_path, 'w', encoding='utf-8') as f:
    json.dump(evaluations, f, ensure_ascii=False, indent=2)
evaluations_path